# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [2]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
from decoder import MyViterbiDecoder
from utils import parse_lexicon, generate_symbol_tables
import math
import time
from collections import Counter

def compute_unigram_probs(wav_files):
    counts = Counter()
    for wav_file in wav_files:
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f:
            words = f.readline().strip().split()
        counts.update(words)
    total = sum(counts.values())
    return {word: count / total for word, count in counts.items()}

def create_wfst(lexicon, word_table, phone_table, state_table, unigram_probs, n_states=3, stay_cost=0.9, final_prob=0.1):
    f = fst.Fst('log')
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    loop_state = f.add_state()
    f.set_start(loop_state)
    f.set_final(loop_state)

    trans_cost = 1.0 - stay_cost

    n_sil_states = 5
    sil_states = [f.add_state() for _ in range(n_sil_states)]

    for i, sil_state in enumerate(sil_states):
        sil_label = state_table.find(f"sil_{i+1}")
        f.add_arc(sil_state, fst.Arc(sil_label, 0, fst.Weight('log', -math.log(stay_cost)), sil_state))
        if i < n_sil_states - 1:
            f.add_arc(sil_state, fst.Arc(sil_label, 0, fst.Weight('log', -math.log(trans_cost)), sil_states[i+1]))
        else:
            f.add_arc(sil_state, fst.Arc(sil_label, 0, fst.Weight('log', -math.log(trans_cost)), loop_state))

    f.add_arc(loop_state, fst.Arc(0, 0, fst.Weight.One(f.weight_type()), sil_states[0]))

    for word, phones in lexicon.items():
        word_id = word_table.find(word)
        current_state = loop_state

        prob = unigram_probs.get(word, 1.0 / len(lexicon))
        unigram_weight = -math.log(prob)

        for phone_idx, phone in enumerate(phones):
            for i in range(1, n_states + 1):
                state_sym = f"{phone}_{i}"
                in_label = state_table.find(state_sym)

                sl_weight = fst.Weight('log', -math.log(stay_cost))
                f.add_arc(current_state, fst.Arc(in_label, 0, sl_weight, current_state))

                next_state = f.add_state()

                if phone_idx == 0 and i == 1:
                    fw_weight = fst.Weight('log', -math.log(trans_cost) + unigram_weight)
                else:
                    fw_weight = fst.Weight('log', -math.log(trans_cost))

                if phone_idx == len(phones) - 1 and i == n_states:
                    out_label = word_id
                else:
                    out_label = 0

                f.add_arc(current_state, fst.Arc(in_label, out_label, fw_weight, next_state))
                current_state = next_state

        final_weight = fst.Weight('log', -math.log(final_prob))
        f.add_arc(current_state, fst.Arc(0, 0, final_weight, loop_state))
        f.add_arc(current_state, fst.Arc(0, 0, final_weight, sil_states[0]))

    return f


def read_transcription(wav_file):
    transcription_file = os.path.splitext(wav_file)[0] + '.txt'
    with open(transcription_file, 'r') as f:
        transcription = f.readline().strip()
    return transcription


lex = parse_lexicon("lexicon.txt")
word_table, phone_table, state_table = generate_symbol_tables(lex)

for i in range(1, 6):
    state_table.add_symbol(f"sil_{i}")

stay_cost = 0.9
final_prob = 0.1

wav_files = glob.glob('/group/teaching/asr/labs/recordings/*.wav')
unigram_probs = compute_unigram_probs(wav_files)

f = create_wfst(lex, word_table, phone_table, state_table, unigram_probs,
                stay_cost=stay_cost, final_prob=final_prob)

best_wer = float('inf')
best_beam = None

# ADDED: beam sweep — float('inf') is the untuned baseline (no pruning)
for beam in [float('inf'), 500, 200, 100, 50, 20, 10, 5]:

    total_substitutions = 0
    total_deletions = 0
    total_insertions = 0
    total_words = 0
    total_forward_computations = 0
    total_decode_time = 0.0
    total_backtrace_time = 0.0

    for wav_file in wav_files:

        decoder = MyViterbiDecoder(f, wav_file, beam=beam)

        t0 = time.perf_counter()
        decoder.decode()
        decode_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        (state_path, words) = decoder.backtrace()
        backtrace_time = time.perf_counter() - t0

        total_decode_time += decode_time
        total_backtrace_time += backtrace_time
        total_forward_computations += decoder.forward_computation_count

        words_str = ' '.join(words)
        transcription = read_transcription(wav_file)
        error_counts = wer.compute_alignment_errors(transcription, words_str)
        word_count = len(transcription.split())

        total_substitutions += error_counts[0]
        total_deletions += error_counts[1]
        total_insertions += error_counts[2]
        total_words += word_count

    overall_wer = (total_substitutions + total_deletions + total_insertions) / total_words

    beam_str = f"{beam}" if beam != float('inf') else "inf (no pruning)"
    print(f"=== Results (beam={beam_str}) ===")
    print(f"  WER                        : {overall_wer:.2%}")
    print(f"  Total substitutions        : {total_substitutions}")
    print(f"  Total deletions            : {total_deletions}")
    print(f"  Total insertions           : {total_insertions}")
    print(f"  Total forward computations : {total_forward_computations}")
    print(f"  Total decode time          : {total_decode_time:.4f}s")
    print(f"  Total backtrace time       : {total_backtrace_time:.4f}s")
    print(f"  Total time                 : {total_decode_time + total_backtrace_time:.4f}s\n")

    if overall_wer < best_wer:
        best_wer = overall_wer
        best_beam = beam

print(f"=== Best: beam={best_beam}, WER={best_wer:.2%} ===")

TypeError: __init__() got an unexpected keyword argument 'beam'